# Simple CNN for MNIST (PyTorch)

## Basic CNN with ~90% test accuracy after just 3 epochs

### How it works

- Input: **28x28** grayscale image (single channel)
- Apply **1 convolution layer** with **8 filters**
- Apply **ReLU** activation
- Apply **MaxPool** → makes image **14x14**
- Flatten the feature map
- Pass through **Linear layer** → 10 outputs (digits 0-9)

### Training setup

- Loss: **CrossEntropyLoss**
- Optimizer: **SGD (lr = 0.01)**
- Epochs: **3**


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(8*14*14, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# load MNIST
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=".", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

# model, loss, optimiser
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# training loop
for epoch in range(3):
    total = 0
    correct = 0
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"epoch {epoch+1}: accuracy = {correct/total:.4f}")

100%|██████████| 9.91M/9.91M [00:22<00:00, 445kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 96.8kB/s]
100%|██████████| 1.65M/1.65M [00:03<00:00, 415kB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


epoch 1: accuracy = 0.8193
epoch 2: accuracy = 0.9019
epoch 3: accuracy = 0.9108
test accuracy: 0.9189


In [2]:
# test accuracy
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("test accuracy:", correct/total)


test accuracy: 0.9189
